# L2 step 3 · The two 3Bs on the held-out questions, and the rule (GPU T4)

Task **L2** (roadmap), objective **O4**. Decided 8 Nov: two 3Bs, **(b)** soft labels and **(c)** the options rerun, and the better one is kept by a rule fixed before any run. This notebook applies it.

**The questions:** the 1,962 checkable training questions on the 501 after-photos that neither 3B trained on (1 in 20, by a hash of the file name), asked in the benchmark's format, with the options, greedy, images at the 3B's training size. The base 3B, untrained, answers them too, as a reference.

**The rule, fixed 8 Nov (roadmap L2)**
- **Primary score:** accuracy, with the two counting tasks limited to questions whose largest option is 20 or less (where the middle-value pattern can't help); the other four tasks use every question.
- If exact McNemar on that set gives p < 0.05, the more accurate 3B wins. Otherwise the lower median count error on all held-out building-counting questions wins (the value of the option each chose, against the true count, on questions both answered with one option); if that is equal too, (b) wins.
- **Reported, not deciding:** accuracy on every question and by task; on counting questions whose largest option is above 20; how often each picks the middle option there; count error; the rule that never sees the image on the same questions. (c) has also seen the benchmark's letter format on three tasks, which the rule can't remove.

**How to run on Kaggle** (account 2, once both trainings are complete)
1. Settings: GPU **T4 x2**, internet ON. Secret `HF_TOKEN` (AbrarAlam, read: the adapters are in the private repo).
2. Input: `abrarmohammedtanzim/disasterm3-mirror`.
3. Run All once with `SMOKE_TEST = True` in Cell 2 (12 questions per model). Then `False` and Save & Run All. About 1–1.5 h for the three models (estimated).
4. Outputs in `/kaggle/working/L2S3/`: `selection.json` (the winner), `per_question.csv`, `scores.csv`, `README.md`.


In [1]:
# ── Cell 1: Environment: Kaggle's torch and transformers, current peft (as L2 step 2), and no torchao ──
# L0's merge (26 Oct) stopped on "Found an incompatible version of torchao": peft checks torchao when it puts an adapter
# on an fp16 model, as this notebook does. Nothing here uses torchao.
import importlib.metadata as _md
import importlib.util
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "peft"], check=True)

def _version(pkg):
    try:
        return _md.version(pkg)
    except _md.PackageNotFoundError:
        return "0"

assert importlib.util.find_spec("torchao") is None, "torchao is still installed: restart the session and run again"
assert int(_version("transformers").split(".")[0]) >= 5, "written for transformers 5 (Kaggle's 5.16.1 on 26 Oct)"
import torch
VERSIONS = {p: _version(p) for p in ("torch", "transformers", "peft")}
print(f"✓ {VERSIONS}, python {sys.version.split()[0]}; torchao: not installed")
print(f"  GPUs: {[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}")
assert torch.cuda.device_count() >= 1, "Pick GPU T4 x2 (one is used)"


✓ {'torch': '2.11.0+cu130', 'transformers': '5.20.2', 'peft': '0.22.0'}, python 3.13.16; torchao: not installed
  GPUs: ['Tesla T4', 'Tesla T4']


In [2]:
# ── Cell 2: Configuration ─────────────────────────────────────────────────
import ast, json, math, os, random, re, time
from pathlib import Path

import numpy as np
import pandas as pd
from huggingface_hub import snapshot_download
from PIL import Image
from scipy.stats import binomtest
from transformers import AutoProcessor

DATA_ROOT = Path("/kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct")
MANIFEST_PATH = DATA_ROOT / "train_release.json"
BASE_MODEL = "Qwen/Qwen2.5-VL-3B-Instruct"
ADAPTER_REPO = "AbrarAlam/disasterm3-qwen25vl3b-l2"      # private: b/final and c/final
MODELS = ("base", "b", "c")
BATCH, MAX_NEW_TOKENS = 8, 32
SMOKE_TEST = True             # True = 12 questions per model
OUT_DIR = Path("/kaggle/working/L2S3")
OUT_DIR.mkdir(parents=True, exist_ok=True)
CUDA = torch.cuda.is_available()

import glob as _g, os as _o
for _f in _g.glob("/kaggle/input/**/hf_write_token.txt", recursive=True):   # runs started from the PC have no secrets
    _o.environ["HF_TOKEN"] = open(_f, encoding="utf-8").read().strip()

def hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return os.environ.get("HF_TOKEN")

# ── The rules all L2 step 2 notebooks share (the same text in each) ─────────
import hashlib

PATH_SUFFIXES = (".png", ".jpg", ".jpeg", ".tif", ".tiff")
CHECKABLE = {"Building Damage Counting", "Road Damage Counting", "Disaster Type Recognition",
             "Disaster Bearing Bodies Recognition", "Disaster Scene Recognition", "relational reasoning"}
MAX_PIXELS = 768 * 28 * 28       # the 3B's image cap: a 1024 px image reaches it at 756 px (729 tokens); also the soft labels'
HELD_OUT_EVERY = 20              # 1 after-photo in 20 is held out to choose between the two 3Bs (fixed 8 Nov)
ELIGIBLE, CHECKABLE_TRAIN, CHECKABLE_HELD_OUT = 55764, 38271, 1962   # counted on the PC from train_release.json, 8 Nov

def name_of(path):
    return Path(str(path).replace("\\", "/")).name

def scene(e):
    """The after-photo a question is about. A relational question's boxed picture, '<after-photo>_<k>.png',
    counts as its after-photo, so the same scene never sits on both sides of the split."""
    if e.get("post_image_path"):
        return name_of(e["post_image_path"])
    return re.sub(r"_\d+(\.png)$", r"\1", name_of(e.get("image_path", "")))

def held_out(e):
    return int(hashlib.md5(scene(e).encode()).hexdigest(), 16) % HELD_OUT_EVERY == 0

def resolve_image_path(rel_path):
    """L2 step 1's rule: a manifest path -> its file in the mirror (None if missing)."""
    if not rel_path:
        return None
    rel_path = rel_path.replace("\\", "/")
    filename = Path(rel_path).name
    for candidate in (DATA_ROOT / rel_path, DATA_ROOT / "train_images" / rel_path, DATA_ROOT / "box_train_images" / rel_path,
                      DATA_ROOT / "masks" / rel_path, DATA_ROOT / "train_images" / "train_images" / filename,
                      DATA_ROOT / "box_train_images" / "box_train_images" / filename, DATA_ROOT / "masks" / "masks" / filename,
                      Path("/tmp") / rel_path):
        if candidate.exists():
            return str(candidate)
    return None

def answer_text(entry):
    ans = entry.get("training_answer", "")
    if isinstance(ans, list):
        ans = ans[0] if ans else ""
    return str(ans).strip()

def eligible(entry):
    """L2 step 1's rule (the 7B's training rule): no segmentation, a text answer, and every image found."""
    if entry.get("task") == "Referring Expression Segmentation":
        return False
    ans = answer_text(entry)
    if not ans or ans.lower().endswith(PATH_SUFFIXES):
        return False
    if "image_path" in entry:
        return resolve_image_path(entry["image_path"]) is not None
    return (resolve_image_path(entry.get("pre_image_path", "")) is not None
            and resolve_image_path(entry.get("post_image_path", "")) is not None)

def smart_resize(height, width, min_pixels, max_pixels, factor=28):
    """Qwen2.5-VL's rule for the size an image reaches the model at (its image processor's smart_resize)."""
    h, w = round(height / factor) * factor, round(width / factor) * factor
    if h * w > max_pixels:
        beta = math.sqrt(height * width / max_pixels)
        h, w = math.floor(height / beta / factor) * factor, math.floor(width / beta / factor) * factor
    elif h * w < min_pixels:
        beta = math.sqrt(min_pixels / (height * width))
        h, w = math.ceil(height * beta / factor) * factor, math.ceil(width * beta / factor) * factor
    return h, w

def sized(img, max_pixels=MAX_PIXELS):
    """The image at the 3B's cap: only an image above it is scaled down (as L0 did at full size)."""
    if img.width * img.height <= max_pixels:
        return img
    h, w = smart_resize(img.height, img.width, 56 * 56, max_pixels)
    return img.resize((w, h), Image.BICUBIC)

assert smart_resize(1024, 1024, 56 * 56, MAX_PIXELS) == (756, 756)
assert held_out({"image_path": "x\\box_train_images\\a_post_7_3.png"}) == held_out({"post_image_path": "a_post_7.png"})
# ── The benchmark's prompts and scorer, and the 3Bs (the same code in steps 3 and 4) ──
# The four prompts, copied from L0 step 2 (which copied them from the DisasterM3 authors' runner)
PROMPTS = dict(
    bearing_body="""Analyze both the pre-disaster and post-disaster images to answer the following question. Choose the best option(s) from the candidate options provided.

pre-disaster image:
<image>

post-disaster image:
<image>

Question: {prompt}
Options: {options_str}

Your task is to respond with ONLY the capital letters of the correct options, separated by a comma and a space (e.g., C, D, H). Do not include any explanation or other text.""",
    landuse="""Analyze the image to answer the following question. Choose the best option(s) from the candidate options provided.

Question: {prompt}
Options: {options_str}

Your task is to respond with ONLY the capital letters of the correct options, separated by a comma and a space (e.g., C, D, H). Do not include any explanation or other text.""",
    single_choice="""Analyze both the pre-disaster and post-disaster images to answer the following question. Choose the best option from the candidate options provided.

pre-disaster image:
<image>

post-disaster image:
<image>

Question: {prompt}
Options: {options_str}

Your task is to respond with ONLY the capital letter of the correct option (e.g., C). Do not include any explanation or other text.""",
    relational_reasoning_qa="""Analyze the image to answer the following question. Choose the best option from the candidate options provided.

Question: {prompt}
Options: {options_str}

Your task is to respond with ONLY the capital letter of the correct option (e.g., C). Do not include any explanation or other text.""",
)
# (code, subset name in the 499A scorer, task name in the benchmark, task name in train_release.json)
TASKS = [("DTR", "disaster_type", "Disaster Type Recognition", "Disaster Type Recognition"),
         ("BBR", "bearing_body", "Disaster Bearing Bodies Recognition", "Disaster Bearing Bodies Recognition"),
         ("BDC", "building_damage_counting", "Building Damage Counting", "Building Damage Counting"),
         ("DRE", "road_damage_counting", "Road Damage Counting", "Road Damage Counting"),
         ("DSR", "landuse", "Disaster Scene Recognition", "Disaster Scene Recognition"),
         ("ORR", "relational_reasoning_qa", "Relational Reasoning", "relational reasoning")]
COUNTING = ("BDC", "DRE")

def letters(text):
    """The 499A scorer: the set of capital letters left after removing 'THE ANSWER IS' and 'OPTION'."""
    t = str(text).upper().replace("THE ANSWER IS", "").replace("OPTION", "")
    return set(re.findall(r"[A-Z]", t))

def prompt_text(e, subset):
    """(the benchmark prompt, the image keys in order), as L0 step 2's build() makes them."""
    p = e.get("prompts", "")
    p = p[0] if isinstance(p, list) else p
    if subset == "landuse":
        return PROMPTS["landuse"].format(prompt=p, options_str=e["options_str"]), ["pre_image_path"]
    if subset == "relational_reasoning_qa":           # its entries spell the options' field option_str
        return PROMPTS[subset].format(prompt=p, options_str=e["option_str"]), ["image_path"]
    key = "bearing_body" if subset == "bearing_body" else "single_choice"
    return PROMPTS[key].format(prompt=p, options_str=e["options_str"]), ["pre_image_path", "post_image_path"]

def content_of(text, n_images):
    """L0 step 2's layout: single-image prompts put the image first; two-image prompts fill each '<image>' gap."""
    parts = text.split("<image>")
    if len(parts) == 1:
        return [{"type": "image"} for _ in range(n_images)] + [{"type": "text", "text": text}]
    content = []
    for n, part in enumerate(parts):
        if part:
            content.append({"type": "text", "text": part})
        if n < len(parts) - 1:
            content.append({"type": "image"})
    return content

def option_values(e):
    """The five option values of a counting question, as numbers ('5.22%' -> 5.22)."""
    o = e["options_list"]
    o = ast.literal_eval(o) if isinstance(o, str) else o
    return [float(str(x).rstrip("%")) for x in o]

def no_image_rule(values):
    """Never sees the image: largest option above 20 -> the middle value, otherwise the second smallest (benchmark check, 7 Nov)."""
    s = sorted(values)
    return "ABCDE"[values.index(s[2] if s[4] > 20 else s[1])]

def chosen_value(answer, values):
    """The value of the one option an answer names; None if it names no option or several."""
    got = letters(answer)
    if len(got) != 1 or next(iter(got)) not in "ABCDE"[:len(values)]:
        return None
    return values["ABCDE".index(next(iter(got)))]

assert letters("The answer is D.") == {"D"} and no_image_rule([231, 323, 277, 139, 185]) == "A"
assert chosen_value("B", [1, 2, 3, 4, 5]) == 2 and chosen_value("B, C", [1, 2, 3, 4, 5]) is None
assert content_of("x<image>y<image>z", 2)[1] == {"type": "image"} and content_of("q", 1)[0] == {"type": "image"}

def load_3b(which):
    """The base 3B in fp16 on one T4; for 'b' or 'c', its finished adapter merged in."""
    from peft import PeftModel
    from transformers import Qwen2_5_VLForConditionalGeneration
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(BASE_MODEL, dtype=torch.float16 if CUDA else torch.float32,
                                                               device_map={"": 0} if CUDA else None, attn_implementation="sdpa")
    if which != "base":
        local = snapshot_download(ADAPTER_REPO, allow_patterns=[f"{which}/final/*"], token=hf_token())
        model = PeftModel.from_pretrained(model, str(Path(local) / which / "final")).merge_and_unload()
    return model.eval()

@torch.inference_mode()
def answer_batch(model, items):
    """items: (prompt text, image paths) -> greedy answers, every sampling setting given, images at the 3B's cap."""
    texts, images = [], []
    for text, paths in items:
        imgs = [sized(Image.open(p).convert("RGB")) for p in paths]
        texts.append(processor.apply_chat_template([{"role": "user", "content": content_of(text, len(imgs))}], tokenize=False,
                                                   add_generation_prompt=True))
        images += imgs
    inputs = processor(text=texts, images=images, padding=True, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, temperature=None, top_p=None, top_k=None,
                         repetition_penalty=1.0)
    return [s.strip() for s in processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)]

def answer_safe(model, items):
    """Halves the batch on out-of-memory or an unreadable image; a question that still fails alone gets '' (scored wrong,
    as L0 step 2 scored its unreadable images)."""
    try:
        return answer_batch(model, items)
    except (torch.cuda.OutOfMemoryError, OSError):
        torch.cuda.empty_cache()
        if len(items) == 1:
            return [""]
        half = len(items) // 2
        return answer_safe(model, items[:half]) + answer_safe(model, items[half:])

processor = AutoProcessor.from_pretrained(BASE_MODEL)
processor.tokenizer.padding_side = "left"
print(f"✓ L2 step 3{'  [SMOKE TEST]' if SMOKE_TEST else ''}: models {MODELS}; images at most {MAX_PIXELS:,} pixels")


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

✓ L2 step 3  [SMOKE TEST]: models ('base', 'b', 'c'); images at most 602,112 pixels


In [3]:
# ── Cell 3: The held-out questions, in the benchmark's format ─────────────────
with open(MANIFEST_PATH, encoding="utf-8") as f:
    raw = json.load(f)
eligible_entries = [e for e in raw if eligible(e)]
del raw
assert len(eligible_entries) == ELIGIBLE
CODE_OF = {train_name: (code, subset) for code, subset, _, train_name in TASKS}
rows = []
for i, e in enumerate(eligible_entries):
    if e["task"] not in CODE_OF or not held_out(e):
        continue
    code, subset = CODE_OF[e["task"]]
    text, keys = prompt_text(e, subset)
    row = {"entry_idx": i, "task": code, "text": text, "images": [resolve_image_path(e[k]) for k in keys],
           "truth": e["ground_truth_option"], "truth_value": None, "largest_option": None, "rule_correct": None, "middle_correct": None}
    if code in COUNTING:
        values = option_values(e)
        right = sorted(letters(e["ground_truth_option"]))[0]
        row.update(truth_value=values["ABCDE".index(right)], largest_option=max(values), values=values,
                   rule_correct=no_image_rule(values) == right, middle_correct=values["ABCDE".index(right)] == sorted(values)[2])
    rows.append(row)
assert len(rows) == CHECKABLE_HELD_OUT, f"{len(rows):,} held-out questions, not {CHECKABLE_HELD_OUT:,}"
Q = pd.DataFrame(rows)
Q["primary"] = ~(Q["task"].isin(COUNTING) & (Q["largest_option"] > 20))
if SMOKE_TEST:
    Q = Q.groupby("task").head(2).reset_index(drop=True)
print(f"✓ {len(Q):,} held-out questions { {k: int(v) for k, v in Q['task'].value_counts().items()} }; "
      f"in the primary score: {int(Q['primary'].sum()):,}")


✓ 12 held-out questions {'BBR': 2, 'BDC': 2, 'DRE': 2, 'DTR': 2, 'DSR': 2, 'ORR': 2}; in the primary score: 10


In [4]:
# ── Cell 4: Each 3B answers every question (the base 3B too, as a reference) ───
import gc
from tqdm.auto import tqdm

ANSWERS, TIMES = {}, {}
for which in MODELS:
    t0 = time.time()
    model = load_3b(which)
    got = []
    for start in tqdm(range(0, len(Q), BATCH), desc=which):
        batch = Q.iloc[start:start + BATCH]
        got += answer_safe(model, list(zip(batch["text"], batch["images"])))
    ANSWERS[which] = got
    TIMES[which] = round((time.time() - t0) / 60, 1)
    print(f"✓ {which}: {len(got):,} answers in {TIMES[which]} min; first: {got[:4]}")
    del model
    gc.collect()
    if CUDA:
        torch.cuda.empty_cache()


model.safetensors.index.json:   0%|          | 0.00/65.4k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

base:   0%|          | 0/2 [00:00<?, ?it/s]

... (7 outputs left out)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

c:   0%|          | 0/2 [00:00<?, ?it/s]

✓ c: 12 answers in 0.4 min; first: ['D, C', 'A, B, G, F', 'E', 'B']


In [5]:
# ── Cell 5: Scores, and the rule ───────────────────────────────────────────
for which in MODELS:
    Q[f"{which} | answer"] = ANSWERS[which]
    Q[f"{which} | correct"] = [letters(a) == letters(t) for a, t in zip(ANSWERS[which], Q["truth"])]
    Q[f"{which} | value"] = [chosen_value(a, v) if task in COUNTING else None
                             for a, v, task in zip(ANSWERS[which], Q.get("values", [None] * len(Q)), Q["task"])]
    Q[f"{which} | middle"] = [(val is not None and val == sorted(v)[2]) if task in COUNTING else None
                              for val, v, task in zip(Q[f"{which} | value"], Q.get("values", [None] * len(Q)), Q["task"])]

def acc(df, which):
    return round(100 * df[f"{which} | correct"].mean(), 2) if len(df) else float("nan")

big = Q["task"].isin(COUNTING) & (Q["largest_option"] > 20)
bdc = Q[Q["task"] == "BDC"]
both = bdc[bdc["b | value"].notna() & bdc["c | value"].notna()]
def median_error(df, which):
    err = (df[f"{which} | value"].astype(float) - df["truth_value"]).abs() / df["truth_value"].clip(lower=1)
    return round(100 * float(err.median()), 1) if len(df) else float("nan")

score_rows = []
for which in MODELS:
    score_rows.append({"model": which, "primary_%": acc(Q[Q["primary"]], which), "all_%": acc(Q, which),
                       **{f"{code}_%": acc(Q[Q["task"] == code], which) for code, _, _, _ in TASKS},
                       "counting_largest_above_20_%": acc(Q[big], which),
                       "middle_picked_above_20_%": round(100 * float(Q.loc[big, f"{which} | middle"].astype(float).mean()), 1) if big.any() else float("nan"),
                       "bdc_median_count_error_%": median_error(both, which)})
scores = pd.DataFrame(score_rows)
counting_rows = Q["task"].isin(COUNTING)
rule_ref = {"no_image_rule_on_counting_%": round(100 * float(Q.loc[counting_rows, "rule_correct"].astype(float).mean()), 2),
            "always_middle_on_counting_%": round(100 * float(Q.loc[counting_rows, "middle_correct"].astype(float).mean()), 2),
            "always_middle_largest_above_20_%": round(100 * float(Q.loc[big, "middle_correct"].astype(float).mean()), 2) if big.any() else None}
print(scores.to_string(index=False))
print(f"For scale, on the same counting questions: {rule_ref}")

# The rule (fixed 8 Nov)
P = Q[Q["primary"]]
only_b, only_c = int((P["b | correct"] & ~P["c | correct"]).sum()), int((~P["b | correct"] & P["c | correct"]).sum())
p = binomtest(only_b, only_b + only_c, 0.5).pvalue if only_b + only_c else 1.0
acc_b, acc_c = acc(P, "b"), acc(P, "c")
err_b, err_c = median_error(both, "b"), median_error(both, "c")
if p < 0.05:
    winner = "b" if acc_b > acc_c else "c"
    reason = f"primary accuracy {acc_b}% (b) against {acc_c}% (c), exact McNemar p = {p:.4f}"
elif err_b != err_c:
    winner = "b" if err_b < err_c else "c"
    reason = (f"primary accuracy not clearly different ({acc_b}% against {acc_c}%, p = {p:.4f}); lower median count error on the "
              f"held-out building-counting questions: {err_b}% (b) against {err_c}% (c), on {len(both)} questions both answered")
else:
    winner = "b"
    reason = f"primary accuracy and count error level ({acc_b}% / {acc_c}%, p = {p:.4f}; {err_b}% / {err_c}%): (b) by the rule"
print(f"\n{'SMOKE TEST, no verdict. ' if SMOKE_TEST else ''}Winner: ({winner}): {reason}")


model  primary_%  all_%  DTR_%  BBR_%  BDC_%  DRE_%  DSR_%  ORR_%  counting_largest_above_20_%  middle_picked_above_20_%  bdc_median_count_error_%
 base       10.0  25.00    0.0    0.0  100.0    0.0   50.0    0.0                        100.0                     100.0                       0.0
    b       10.0  16.67    0.0    0.0   50.0    0.0   50.0    0.0                         50.0                      50.0                      19.6
    c       50.0  41.67    0.0    0.0    0.0  100.0   50.0  100.0                          0.0                       0.0                      18.2
For scale, on the same counting questions: {'no_image_rule_on_counting_%': 50.0, 'always_middle_on_counting_%': 75.0, 'always_middle_largest_above_20_%': 100.0}

SMOKE TEST, no verdict. Winner: (c): primary accuracy not clearly different (10.0% against 50.0%, p = 0.1250); lower median count error on the held-out building-counting questions: 19.6% (b) against 18.2% (c), on 2 questions both answered


In [6]:
# ── Cell 6: The output files ─────────────────────────────────────────────────
def as_markdown(df):
    try:
        return df.to_markdown(index=False)
    except ImportError:
        return "```\n" + df.to_string(index=False) + "\n```"

Q.drop(columns=["text"]).to_csv(OUT_DIR / "per_question.csv", index=False)
scores.to_csv(OUT_DIR / "scores.csv", index=False)
selection = {"winner": None if SMOKE_TEST else winner, "reason": reason, "smoke_test": SMOKE_TEST, "adapter": f"{ADAPTER_REPO}/{winner}/final",
             "primary": {"b_%": acc_b, "c_%": acc_c, "only_b_right": only_b, "only_c_right": only_c, "p": round(float(p), 4)},
             "bdc_median_count_error_%": {"b": err_b, "c": err_c, "questions": len(both)}, "references": rule_ref,
             "questions": len(Q), "minutes": TIMES, "versions": VERSIONS}
(OUT_DIR / "selection.json").write_text(json.dumps(selection, indent=1), encoding="utf-8")
(OUT_DIR / "README.md").write_text("\n".join([
    "# L2 step 3: the two 3Bs on the held-out questions" + (" (SMOKE TEST)" if SMOKE_TEST else ""), "",
    f"{len(Q):,} checkable training questions on the 501 after-photos neither 3B trained on, in the benchmark's format, greedy. "
    "The base 3B is a reference.", "",
    f"**Winner by the rule fixed on 8 Nov: ({winner}).** {reason}." if not SMOKE_TEST else "**Smoke test: no verdict.**", "",
    "## Scores", "", as_markdown(scores), "",
    f"- `primary_%`: the rule's score: counting only where the largest option is 20 or less, every question of the other tasks.",
    "- `counting_largest_above_20_%` and `middle_picked_above_20_%`: where the options are built around the correct count, so the "
    "middle value is always right. A high middle share with no gain elsewhere means the pattern, not the picture.",
    f"- On the same counting questions, the rule that never sees the image scores {rule_ref['no_image_rule_on_counting_%']}% and "
    f"always the middle value {rule_ref['always_middle_on_counting_%']}%.",
    "- `bdc_median_count_error_%`: the chosen option's value against the true count, on the building-counting questions both 3Bs "
    "answered with one option.", "",
    "## Files", "", "- `selection.json`, `scores.csv`, `per_question.csv`",
]) + "\n", encoding="utf-8")
print(f"✓ Wrote {OUT_DIR}: selection.json, scores.csv, per_question.csv, README.md")


✓ Wrote /kaggle/working/L2S3: selection.json, scores.csv, per_question.csv, README.md
